# Rolling Feature Dataset Analysis
Analysis of `rolling_feature_engineered_tier_1_games.csv` to evaluate predictive power and data quality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load data
df = pd.read_csv("../../data/rolling_feature_engineered_tier_1_games.csv")
print(f"Dataset shape: {df.shape}")

## 1. Rolling Feature Distributions
Checking player-level rolling averages.

In [ ]:
# find all rolling player columns
player_cols = [col for col in df.columns if "previous_10_game" in col and "player" in col]
print(f"Found {len(player_cols)} player-level rolling features.")

# summary of kills across all player slots
kill_cols = [col for col in player_cols if "average_kills" in col]
kill_summary = df[kill_cols].melt()
print("\nRolling Kills Summary:")
print(kill_summary['value'].describe())

# plot distribution of ADR
adr_cols = [col for col in player_cols if "average_adr" in col]
plt.figure(figsize=(10, 5))
df[adr_cols].mean(axis=1).hist(bins=30)
plt.title("Distribution of Match Average Rolling ADR (All Players)")
plt.xlabel("ADR")
plt.ylabel("Frequency")
plt.show()

## 2. Feature-Target Correlation
Which features predict `team1_win`?

In [ ]:
# Ensure only numeric columns are used for correlation
numeric_df = df.select_dtypes(include=[np.number])
correlations = numeric_df.corr()['team1_win'].sort_values(ascending=False)

print("Top 10 Features Correlated with Team 1 Win:")
print(correlations.head(11)) # include target itself

print("\nBottom 10 Features (Correlated with Team 2 Win):")
print(correlations.tail(10))

## 3. Team Aggregate Comparison
Calculating 'Delta' stats between teams.

In [ ]:
# create aggregate ADR features
t1_adr_cols = [col for col in adr_cols if "team1" in col]
t2_adr_cols = [col for col in adr_cols if "team2" in col]

df['t1_sum_adr'] = df[t1_adr_cols].sum(axis=1)
df['t2_sum_adr'] = df[t2_adr_cols].sum(axis=1)
df['adr_delta'] = df['t1_sum_adr'] - df['t2_sum_adr']

print(f"Correlation of ADR Delta with win: {df['adr_delta'].corr(df['team1_win']):.4f}")

plt.figure(figsize=(10, 5))
sns.boxplot(x='team1_win', y='adr_delta', data=df)
plt.title("ADR Delta vs Team 1 Win")
plt.ylabel("ADR Delta (Team 1 - Team 2)")
plt.show()

## 4. Cold Start Analysis
Checking for matches where players might have low history.

In [ ]:
# checking for zeros or very low ADR which might indicate default values
low_history_mask = (df[adr_cols] == 0).any(axis=1)
print(f"Matches with at least one player having 0 rolling ADR: {low_history_mask.sum()}")
print(f"Percentage of dataset: {low_history_mask.mean()*100:.2f}%")

## 5. Score Validation
Win rate vs historical map score.

In [ ]:
# bin the map score to see win rates
df['score_bin'] = pd.cut(df['team1_previous_10_average_map_score'], bins=5)
win_rate_by_score = df.groupby('score_bin', observed=True)['team1_win'].mean()

print("Win Rate by Team 1 Historical Map Score (Last 10):")
print(win_rate_by_score)

plt.figure(figsize=(10, 5))
win_rate_by_score.plot(kind='bar')
plt.title("Win Rate by Team 1 Historical Avg Score")
plt.ylabel("Win Rate")
plt.xticks(rotation=45)
plt.show()